# House Price Prediction
- **목표**
  - 이 워크샵은 주어진 데이터셋을 이용해 심층신경망 모델을 학습시켜 주택의 최종 판매 가격(SalePrice)을 예측하는 것이 최종 목표입니다.

- **데이터셋 정보**
  - 데이터셋은 총 79개의 설명 변수와 타겟 변수인 주택 가격(SalePrice)로 구성됩니다.
  - 설명 변수는 주택의 다양한 특성(예: 건축 연도, 면적, 위치, 방 개수 등)을 포함합니다.
  - 데이터는 판매 가격이 포함된 학습용 데이터인 `X`, `y` 와 판매 가격이 포함되지 않은 평가용 데이터인 `TEST`파일로 나뉘며, 각각 모델 학습 및 평가에 사용됩니다.
    - 평가용 데이터 `TEST`의 판매 가격(SalePrice)를 예측 후 리더보드로 제출하여 평가합니다.

- **문제 유형**
  - 이 워크샵은 회귀 문제로 연속형 변수를 예측하는 것이 목표입니다. 모델의 성능은 `Mean Absolute Error`로 측정됩니다.

## 1. 환경 설정

In [1]:
%%capture
!pip install JAEN -U

In [2]:
# 그대로 실행하세요.
import torch
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [3]:
# 사용자명을 입력하세요. (이름이 아니여도 괜찮습니다.)
username = "브라운"
assert username, "username 변수에 값이 설정되지 않았습니다."

In [4]:
# 그대로 실행하세요.
from JAEN.competition import Competition
comp = Competition(
    username=username,
    course_name='AI Essential',
    course_round='0317(4)',
    competition_name='House Price Prediction'
)

## 2. 데이터 로드

In [5]:
from JAEN.datasets import load_house_price
X, y, TEST = load_house_price()
X.shape, y.shape, TEST.shape

(torch.Size([1460, 79]), torch.Size([1460, 1]), torch.Size([1459, 79]))

## 3. 제출 예시 코드

In [6]:
# TEST의 예측값 대입 (지금은 0으로 채워진 값 대입)
comp.prediction =  torch.zeros(1459)
comp.prediction

tensor([0., 0., 0.,  ..., 0., 0., 0.])

In [7]:
# 제출
comp.submit()

[House Price Prediction 평가 결과]
 AI Essential 0317(4) 과정 브라운님의 점수는 180493.328125 입니다.

## 4. 심층신경망 모델을 구성하고 학습하여 TEST를 예측해보세요.
- TEST의 예측 결과는 `comp.prediction`에 대입해주세요. **torch.tensor** 형태여야합니다.

In [8]:
# DataLoader 생성
from torch.utils.data import DataLoader, TensorDataset
train_loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

In [9]:
# DNN 모델 구성
class DNN(nn.Module):
    def __init__(self):
        super().__init__()
        # input features를 79로 변경
        self.fc1 = nn.Linear(79, 64)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(32, 1)
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

model = DNN().to(device)
summary(model)

Layer (type:depth-idx)                   Param #
DNN                                      --
├─Linear: 1-1                            5,120
├─ReLU: 1-2                              --
├─Linear: 1-3                            2,080
├─ReLU: 1-4                              --
├─Linear: 1-5                            33
Total params: 7,233
Trainable params: 7,233
Non-trainable params: 0

In [10]:
# 손실함수 및 옵티마이저 설정
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [11]:
X.shape

torch.Size([1460, 79])

In [12]:
# 모델 학습 과정 구현

epochs = 10
for epoch in range(epochs):

    running_loss = 0.0

    # 모델 학습 모드 설정
    model.train()

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)


        # 순전파 > outputs
        outputs = model(inputs)

        # 손실 계산 > loss
        loss = criterion(outputs, labels)

        # 누적된 기울기 초기화
        optimizer.zero_grad()

        # 역전파
        loss.backward()

        # 파라미터 업데이트
        optimizer.step()


        running_loss += loss.item()

    if epoch % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {running_loss / len(train_loader)}')

Epoch [1/10], Loss: 39096957773.91304


In [14]:
# 학습된 모델의 TEST 예측

epochs = 10
for epoch in range(10): # epochs 변수의 값 만큼 반복
    running_loss = 0.0

    # 모델을 학습모드로 설정
    model.train()

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        # inputs에 대한 출력값(outputs) 계산
        outputs = model(inputs)

        # 출력값(outputs)과 정답값(labels)의 손실 계산
        loss = criterion(outputs, labels)

        # 기울기 초기화
        optimizer.zero_grad()

        # 손실(loss)에 대한 각 파라미터의 기울기(gradient)를 계산
        loss.backward()

        # 계산된 기울기를 사용하여 각 파라미터를 업데이트
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch {epoch+1}, Loss: {running_loss / len(train_loader)}') # 에폭별 손실 값 출력
# ================================================================== #

Epoch 1, Loss: 35696511866.434784
Epoch 2, Loss: 34801530701.91304
Epoch 3, Loss: 33692246594.782608
Epoch 4, Loss: 32331587717.565216
Epoch 5, Loss: 30860809527.652172
Epoch 6, Loss: 29295037172.869564
Epoch 7, Loss: 27660943716.173912
Epoch 8, Loss: 25967823916.52174
Epoch 9, Loss: 24175687123.47826
Epoch 10, Loss: 22357293367.652172


In [19]:
# TEST 데이터에 대한 예측 수행
model.eval()  # 모델을 평가 모드로 설정
with torch.no_grad():  # 기울기 계산 비활성화
    predictions = model(TEST.to(device))  # TEST 데이터에 대한 예측 수행

# comp.prediction에 TEST 예측 결과 대입
comp.prediction = predictions.squeeze() # prediction 크기 조정

In [20]:
# 제출
comp.submit()

[House Price Prediction 평가 결과]
 AI Essential 0317(4) 과정 브라운님의 점수는 135843.046875 입니다.